# LARNet Complete Training + Ablation Study (GPU)

This notebook runs the remaining experiments on GPU:
1. **LARNet main model** - 100 epochs (skipped in this run; already completed)
2. **Ablation: Loss + Activation** - 20 epochs each (6 experiments)
3. **Ablation: Filter Width + Optimizer** - 20 epochs each (6 experiments)

In [ ]:
import os
os.environ.setdefault('OMP_NUM_THREADS', '1')
os.environ.setdefault('MKL_NUM_THREADS', '1')
os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')
os.environ.setdefault('NUMEXPR_NUM_THREADS', '1')

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
import time
import json

DATALOADER_WORKERS = int(os.environ.get('DATALOADER_WORKERS', '12'))
DATALOADER_PREFETCH = int(os.environ.get('DATALOADER_PREFETCH', '4'))
TRAIN_BATCH_SIZE = int(os.environ.get('TRAIN_BATCH_SIZE', '128'))
ABLATION_EPOCHS = int(os.environ.get('ABLATION_EPOCHS', '20'))

print(f'CUDA_VISIBLE_DEVICES={os.environ.get("CUDA_VISIBLE_DEVICES", "not set")}')
print(f'DATALOADER_WORKERS={DATALOADER_WORKERS}, PREFETCH={DATALOADER_PREFETCH}, BATCH={TRAIN_BATCH_SIZE}, ABLATION_EPOCHS={ABLATION_EPOCHS}')
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    torch.backends.cudnn.benchmark = True
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    props = torch.cuda.get_device_properties(0)
    print(f'Memory: {props.total_memory / 1e9:.1f} GB')

torch.set_num_threads(1)
os.makedirs('results', exist_ok=True)
print('Setup complete!')

## Data Loading

In [ ]:
def _seed_worker(worker_id):
    torch.set_num_threads(1)


def _loader_kwargs(num_workers):
    kwargs = {
        'num_workers': num_workers,
        'pin_memory': device.type == 'cuda',
    }
    if num_workers > 0:
        kwargs.update({
            'persistent_workers': True,
            'prefetch_factor': DATALOADER_PREFETCH,
            'worker_init_fn': _seed_worker,
        })
    return kwargs


def get_data(batch_size=TRAIN_BATCH_SIZE, num_workers=DATALOADER_WORKERS):
    transform_train = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ])
    transform_test = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ])
    trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
    testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)
    trainloader = torch.utils.data.DataLoader(
        trainset,
        batch_size=batch_size,
        shuffle=True,
        **_loader_kwargs(num_workers),
    )
    testloader = torch.utils.data.DataLoader(
        testset,
        batch_size=256,
        shuffle=False,
        **_loader_kwargs(num_workers),
    )
    return trainloader, testloader

trainloader, testloader = get_data()
print(f'Training: {len(trainloader.dataset)} samples, {len(trainloader)} batches')
print(f'Test: {len(testloader.dataset)} samples')

## Model Definitions

In [ ]:
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        self.squeeze = nn.AdaptiveAvgPool2d(1)
        self.excitation = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )
    def forward(self, x):
        b, c, _, _ = x.shape
        w = self.squeeze(x).view(b, c)
        w = self.excitation(w).view(b, c, 1, 1)
        return x * w

class DepthwiseSeparableConv(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, act_fn=None):
        super().__init__()
        self.depthwise = nn.Conv2d(in_channels, in_channels, 3, stride, 1, groups=in_channels, bias=False)
        self.pointwise = nn.Conv2d(in_channels, out_channels, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(in_channels)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.act = act_fn if act_fn else nn.ReLU(inplace=True)
    def forward(self, x):
        x = self.act(self.bn1(self.depthwise(x)))
        x = self.act(self.bn2(self.pointwise(x)))
        return x

class LARBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1, act_fn=None):
        super().__init__()
        self.conv = DepthwiseSeparableConv(in_ch, out_ch, stride, act_fn)
        self.se = SEBlock(out_ch)
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride, bias=False),
                nn.BatchNorm2d(out_ch)
            )
        else:
            self.shortcut = nn.Identity()
        self.act = act_fn if act_fn else nn.ReLU(inplace=True)
    def forward(self, x):
        return self.act(self.se(self.conv(x)) + self.shortcut(x))

# Full LARNet (for main training)
class LARNet(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 32, 3, 1, 1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True)
        )
        self.stage1 = nn.Sequential(LARBlock(32, 64), LARBlock(64, 64))
        self.stage2 = nn.Sequential(LARBlock(64, 128, stride=2), LARBlock(128, 128), LARBlock(128, 128))
        self.stage3 = nn.Sequential(LARBlock(128, 256, stride=2), LARBlock(256, 256), LARBlock(256, 256))
        self.stage4 = nn.Sequential(LARBlock(256, 512, stride=2), LARBlock(512, 512))
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(0.2)
        self.fc = nn.Linear(512, num_classes)
    def forward(self, x):
        x = self.stem(x)
        x = self.stage1(x)
        x = self.stage2(x)
        x = self.stage3(x)
        x = self.stage4(x)
        return self.fc(self.dropout(self.pool(x).flatten(1)))

# Variable-width LARNet (for ablation)
class LARNetVar(nn.Module):
    def __init__(self, channels, num_classes=10, act_name='ReLU'):
        super().__init__()
        acts = {'ReLU': nn.ReLU(inplace=True), 'LeakyReLU': nn.LeakyReLU(0.1, inplace=True), 'GELU': nn.GELU()}
        act_fn = acts.get(act_name, nn.ReLU(inplace=True))
        c0, c1, c2, c3 = channels
        self.stem = nn.Sequential(nn.Conv2d(3, c0, 3, 1, 1, bias=False), nn.BatchNorm2d(c0), act_fn)
        self.stage1 = LARBlock(c0, c1, stride=1, act_fn=act_fn)
        self.stage2 = LARBlock(c1, c2, stride=2, act_fn=act_fn)
        self.stage3 = LARBlock(c2, c3, stride=2, act_fn=act_fn)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(c3, num_classes)
    def forward(self, x):
        x = self.stem(x)
        x = self.stage1(x)
        x = self.stage2(x)
        x = self.stage3(x)
        return self.fc(self.dropout(self.pool(x).flatten(1)))

def count_params(model):
    return sum(p.numel() for p in model.parameters())

print(f'LARNet params: {count_params(LARNet()):,}')
print(f'LARNetVar [32,64,128,256] params: {count_params(LARNetVar([32,64,128,256])):,}')
print('Models defined!')

## Training Utilities

In [ ]:
def evaluate(model, testloader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in testloader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            outputs = model(images)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)
    return 100.0 * correct / total


def train_model(model, trainloader, testloader, optimizer, scheduler, criterion, epochs, name=''):
    model = model.to(device)
    train_losses, test_accs, epoch_times = [], [], []
    best_acc = 0
    
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        t0 = time.time()
        for images, labels in trainloader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            optimizer.zero_grad()
            outputs = model(images)
            # Handle MSE loss
            if isinstance(criterion, nn.MSELoss):
                targets = torch.zeros(labels.size(0), 10, device=device)
                targets.scatter_(1, labels.unsqueeze(1), 1.0)
                loss = criterion(outputs, targets)
            else:
                loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        scheduler.step()
        elapsed = time.time() - t0
        epoch_times.append(elapsed)
        avg_loss = running_loss / len(trainloader)
        train_losses.append(avg_loss)
        acc = evaluate(model, testloader)
        test_accs.append(acc)
        if acc > best_acc:
            best_acc = acc
        if (epoch+1) % 10 == 0 or epoch == 0 or epoch == epochs-1:
            print(f'  [{name}] Epoch {epoch+1}/{epochs} Loss={avg_loss:.4f} Acc={acc:.2f}% Time={elapsed:.1f}s')
    
    return {
        'name': name, 'train_losses': train_losses, 'test_accs': test_accs,
        'epoch_times': epoch_times, 'best_acc': best_acc, 'final_acc': test_accs[-1],
        'params': count_params(model)
    }

print('Training utilities ready!')

## Task 1: LARNet Main Training (100 epochs)

This is the main model. Target: 92-95% accuracy.

In [ ]:
print('='*70)
print('  TASK 1: LARNet Main Training - 100 epochs SKIPPED')
print('='*70)
result_main = None
print('The 100-epoch main experiment was already completed earlier; skipping it in this run.')

In [ ]:
print('Skipped results/larnet_100ep_curves.png generation because result_main is not available in this run.')

## Task 2: Ablation - Loss Function & Activation (20 epochs each)

In [ ]:
print('='*70)
print(f'  TASK 2A: Loss Function Comparison (ReLU activation, {ABLATION_EPOCHS} epochs)')
print('='*70)

loss_configs = [
    ('CrossEntropy', nn.CrossEntropyLoss()),
    ('MSE', nn.MSELoss()),
    ('LabelSmoothing', nn.CrossEntropyLoss(label_smoothing=0.1)),
]

loss_results = []
for name, loss_fn in loss_configs:
    model = LARNetVar([32, 64, 128, 256], act_name='ReLU').to(device)
    opt = optim.SGD(model.parameters(), lr=0.05, momentum=0.9, weight_decay=5e-4)
    sch = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=ABLATION_EPOCHS)
    result = train_model(model, trainloader, testloader, opt, sch, loss_fn, epochs=ABLATION_EPOCHS, name=f'Loss_{name}')
    loss_results.append(result)

print()
print('='*70)
print(f'  TASK 2B: Activation Function Comparison (CrossEntropy, {ABLATION_EPOCHS} epochs)')
print('='*70)

act_configs = ['ReLU', 'LeakyReLU', 'GELU']
act_results = []
for act_name in act_configs:
    model = LARNetVar([32, 64, 128, 256], act_name=act_name).to(device)
    opt = optim.SGD(model.parameters(), lr=0.05, momentum=0.9, weight_decay=5e-4)
    sch = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=ABLATION_EPOCHS)
    criterion = nn.CrossEntropyLoss()
    result = train_model(model, trainloader, testloader, opt, sch, criterion, epochs=ABLATION_EPOCHS, name=f'Act_{act_name}')
    act_results.append(result)

print()
print('Done! Loss + Activation ablation complete.')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Loss comparison
for r in loss_results:
    epochs_axis = range(1, len(r['train_losses']) + 1)
    axes[0,0].plot(epochs_axis, r['train_losses'], 'o-', label=r['name'].replace('Loss_',''))
axes[0,0].set_xlabel('Epoch'); axes[0,0].set_ylabel('Loss')
axes[0,0].set_title('Loss Function: Training Loss'); axes[0,0].legend(); axes[0,0].grid(True, alpha=0.3)

for r in loss_results:
    epochs_axis = range(1, len(r['test_accs']) + 1)
    axes[0,1].plot(epochs_axis, r['test_accs'], 's-', label=r['name'].replace('Loss_',''))
axes[0,1].set_xlabel('Epoch'); axes[0,1].set_ylabel('Accuracy (%)')
axes[0,1].set_title('Loss Function: Test Accuracy'); axes[0,1].legend(); axes[0,1].grid(True, alpha=0.3)

# Activation comparison
for r in act_results:
    epochs_axis = range(1, len(r['train_losses']) + 1)
    axes[1,0].plot(epochs_axis, r['train_losses'], 'o-', label=r['name'].replace('Act_',''))
axes[1,0].set_xlabel('Epoch'); axes[1,0].set_ylabel('Loss')
axes[1,0].set_title('Activation: Training Loss'); axes[1,0].legend(); axes[1,0].grid(True, alpha=0.3)

for r in act_results:
    epochs_axis = range(1, len(r['test_accs']) + 1)
    axes[1,1].plot(epochs_axis, r['test_accs'], 's-', label=r['name'].replace('Act_',''))
axes[1,1].set_xlabel('Epoch'); axes[1,1].set_ylabel('Accuracy (%)')
axes[1,1].set_title('Activation: Test Accuracy'); axes[1,1].legend(); axes[1,1].grid(True, alpha=0.3)

plt.suptitle(f'Ablation Study: Loss & Activation Comparison ({ABLATION_EPOCHS} epochs)', fontsize=14)
plt.tight_layout()
plt.savefig('results/ablation_loss_activation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/ablation_loss_activation.png')

## Task 3: Ablation - Filter Width & Optimizer (20 epochs each)

In [ ]:
print('='*70)
print(f'  TASK 3A: Filter Width Comparison ({ABLATION_EPOCHS} epochs)')
print('='*70)

filter_configs = {
    'Narrow [16,32,64,128]': [16, 32, 64, 128],
    'Base [32,64,128,256]': [32, 64, 128, 256],
    'Wide [64,128,256,512]': [64, 128, 256, 512],
}

filter_results = []
for name, channels in filter_configs.items():
    model = LARNetVar(channels).to(device)
    print(f'  {name}: {count_params(model):,} params')
    opt = optim.SGD(model.parameters(), lr=0.05, momentum=0.9, weight_decay=5e-4)
    sch = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=ABLATION_EPOCHS)
    criterion = nn.CrossEntropyLoss()
    result = train_model(model, trainloader, testloader, opt, sch, criterion, epochs=ABLATION_EPOCHS, name=name)
    filter_results.append(result)

print()
print('='*70)
print(f'  TASK 3B: Optimizer Comparison ({ABLATION_EPOCHS} epochs)')
print('='*70)

optim_configs = {
    'SGD (lr=0.05)': lambda p: optim.SGD(p, lr=0.05, momentum=0.9, weight_decay=5e-4),
    'Adam (lr=1e-3)': lambda p: optim.Adam(p, lr=1e-3, weight_decay=5e-4),
    'AdamW (lr=1e-3)': lambda p: optim.AdamW(p, lr=1e-3, weight_decay=5e-4),
}

optim_results = []
for name, opt_fn in optim_configs.items():
    model = LARNetVar([32, 64, 128, 256]).to(device)
    optimizer = opt_fn(model.parameters())
    sch = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=ABLATION_EPOCHS)
    criterion = nn.CrossEntropyLoss()
    result = train_model(model, trainloader, testloader, optimizer, sch, criterion, epochs=ABLATION_EPOCHS, name=name)
    optim_results.append(result)

print()
print('Done! Filter + Optimizer ablation complete.')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for r in filter_results:
    epochs_axis = range(1, len(r['train_losses']) + 1)
    axes[0,0].plot(epochs_axis, r['train_losses'], 'o-', label=f"{r['name']} ({r['params']//1000}K)")
axes[0,0].set_xlabel('Epoch'); axes[0,0].set_ylabel('Loss')
axes[0,0].set_title('Filter Width: Training Loss'); axes[0,0].legend(); axes[0,0].grid(True, alpha=0.3)

for r in filter_results:
    epochs_axis = range(1, len(r['test_accs']) + 1)
    axes[0,1].plot(epochs_axis, r['test_accs'], 's-', label=f"{r['name']} ({r['params']//1000}K)")
axes[0,1].set_xlabel('Epoch'); axes[0,1].set_ylabel('Accuracy (%)')
axes[0,1].set_title('Filter Width: Test Accuracy'); axes[0,1].legend(); axes[0,1].grid(True, alpha=0.3)

for r in optim_results:
    epochs_axis = range(1, len(r['train_losses']) + 1)
    axes[1,0].plot(epochs_axis, r['train_losses'], 'o-', label=r['name'])
axes[1,0].set_xlabel('Epoch'); axes[1,0].set_ylabel('Loss')
axes[1,0].set_title('Optimizer: Training Loss'); axes[1,0].legend(); axes[1,0].grid(True, alpha=0.3)

for r in optim_results:
    epochs_axis = range(1, len(r['test_accs']) + 1)
    axes[1,1].plot(epochs_axis, r['test_accs'], 's-', label=r['name'])
axes[1,1].set_xlabel('Epoch'); axes[1,1].set_ylabel('Accuracy (%)')
axes[1,1].set_title('Optimizer: Test Accuracy'); axes[1,1].legend(); axes[1,1].grid(True, alpha=0.3)

plt.suptitle(f'Ablation Study: Filter Width & Optimizer ({ABLATION_EPOCHS} epochs)', fontsize=14)
plt.tight_layout()
plt.savefig('results/ablation_filters_optimizers.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/ablation_filters_optimizers.png')

## Summary & Download Results

In [ ]:
# Summary table
print('='*80)
print(f'{"Experiment":<35} {"Params":<10} {"Best Acc":<10} {"Final Acc":<10} {"Time/Ep":<10}')
print('-'*80)
if result_main is not None:
    print(f'{"LARNet Main (100ep)":<35} {result_main["params"]:>8,}  {result_main["best_acc"]:.2f}%    {result_main["final_acc"]:.2f}%    {np.mean(result_main["epoch_times"]):.1f}s')
    print('-'*80)
else:
    print(f'{"LARNet Main (100ep)":<35} {"SKIPPED":>8}  {"-":>8}    {"-":>8}    {"-":>8}')
    print('-'*80)
for r in loss_results + act_results:
    print(f'{r["name"]:<35} {r["params"]:>8,}  {r["best_acc"]:.2f}%    {r["final_acc"]:.2f}%    {np.mean(r["epoch_times"]):.1f}s')
print('-'*80)
for r in filter_results + optim_results:
    print(f'{r["name"]:<35} {r["params"]:>8,}  {r["best_acc"]:.2f}%    {r["final_acc"]:.2f}%    {np.mean(r["epoch_times"]):.1f}s')
print('='*80)

# Save all results as JSON
all_results = {
    'main_training': result_main,
    'loss_comparison': loss_results,
    'activation_comparison': act_results,
    'filter_comparison': filter_results,
    'optimizer_comparison': optim_results,
}
with open('results/all_results.json', 'w') as f:
    json.dump(all_results, f, indent=2)
print()
print('All results saved to results/all_results.json')
print()
print('Files in results/:')
for f in os.listdir('results'):
    size = os.path.getsize(f'results/{f}')
    print(f'  {f} ({size/1024:.1f} KB)')

In [ ]:
# Download all results as zip
import shutil
shutil.make_archive('LARNet_results', 'zip', 'results')

try:
    from google.colab import files
    files.download('LARNet_results.zip')
    print('Downloading LARNet_results.zip...')
except ImportError:
    print('Saved local archive: LARNet_results.zip')

# Explicitly stop persistent DataLoader workers for local/script execution.
def _shutdown_loader(loader):
    iterator = getattr(loader, '_iterator', None)
    if iterator is not None:
        iterator._shutdown_workers()
        loader._iterator = None

for _loader_name in ('trainloader', 'testloader'):
    _loader = globals().get(_loader_name)
    if _loader is not None:
        _shutdown_loader(_loader)
print('DataLoader workers stopped.')